In [12]:
import pandas as pd
from CCA_utils import *

## Baseline Model Panel

In [13]:
study_sovereigns = [
    'Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']

cca_panel_df = pd.read_csv('../data/processed/CCA/cca_newfx_rates.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()
cca_panel_df.drop(columns=['cds_spread_1Y'], inplace=True)
cca_panel_df.rename(columns={'cds_spread_5Y': 'cds_spread'}, inplace=True)

T=1.0
vol_window = 52
freq = 'W'

cca_panel_df['domestic_rate_in_units'] = cca_panel_df['domestic_rate_in_units']/100
cca_panel_df['risk_free_rate'] = cca_panel_df['risk_free_rate']/100
cca_panel_df['monetary_base_mn_localcurr'] = cca_panel_df['monetary_base_mn_localcurr'] / 1000
cca_panel_df['domestic_debt_bn_localcurr'] = cca_panel_df['domestic_debt_bn_localcurr']
cca_panel_df['external_debt_mn_usd'] = cca_panel_df['external_debt_mn_usd']/1000

## M2 Specific data

In [14]:
ovx_df = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')
ovx_df['date'] = pd.to_datetime(ovx_df['date'])
cca_panel_df = cca_panel_df.merge(ovx_df[['date','OVXCLS']], on='date', how='left')

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)

cca_panel_df['OVXCLS'] = cca_panel_df.groupby('country')['OVXCLS'].ffill()

In [15]:
cca_panel_df['OVXCLS'].isna().sum()

np.int64(0)

In [16]:
import numpy as np
from scipy.optimize import curve_fit

# --- Logistic coefficients from your estimation notebook ---
# lambda_t = lam_max / (1 + exp(-b_fit - a_fit * OVX))  [Eq. logistic, Section 4.3.3]
a_fit   =  0.0780    # beta_1: slope on OVX
b_fit   = -4.0223    # beta_0: intercept
lam_max =  2.0       # FILL: upper bound on annualised jump intensity from your estimation

# Single-parameter eta: mu_J = -eta, sigma_J = eta  [Section 4.3.2]
eta = 0.0806         # mean abs worst daily log-return on Brent within 21-day window

def ovx_to_jump_params(ovx):
    """
    Maps OVX to annualised Poisson intensity via logistic, and returns
    jump size parameters satisfying the single-parameter constraint mu_J = -eta, sigma_J = eta.
    """
    if np.isnan(ovx):
        return 0.0, 0.0, 0.0
    lam = lam_max / (1.0 + np.exp(-b_fit - a_fit * ovx))
    return lam, -eta, eta

# Initialise pricer with 20-term truncation per Section 4.3.4
pricer = AnalyticalJumpDiffusionPricer(max_jumps=20)

results = pd.DataFrame()

print("Starting M2 JD CCA Calibration loop...")

for country, group in cca_panel_df.groupby('country'):
    print(f"Processing {country}...")
    df = group.copy().sort_values('date').reset_index(drop=True)

    r_d     = df['domestic_rate_in_units']
    r_f     = df['risk_free_rate']
    M_bn    = df['monetary_base_mn_localcurr']
    dom_D_bn = df['domestic_debt_bn_localcurr']
    ext_D_bn = df['external_debt_mn_usd']
    fx_rate  = df['fx_rate']

    # --- LCL$ ---
    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(M_bn, dom_D_bn, fx_rate, r_d, r_f)
    ]

    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    # --- Barrier ---
    df['B_f'] = [
        compute_barrier_kvm(debt, rf, T)
        for debt, rf in zip(ext_D_bn, r_f)
    ]

    out = {'implied_V': [], 'implied_sigma_V': [], 'cca_converged': [],
           'distance_to_distress': [], 'default_prob': [],
           'model_spread_bps': [], 'put_value': [], 'risky_debt': [],
           'leverage': []}

    # --- Solve CCA for each week ---
    for i, row in df.iterrows():
        # OVX -> annualised jump intensity and single-parameter jump sizes
        lam_ovx_ann, mu_ovx, sigma_ovx = ovx_to_jump_params(row['OVXCLS'])

        # A. Warm-start guess from baseline GBM solver
        cca_base = solve_CCA(row['LCL_usd'], row['sigma_lcl'], row['B_f'], r_f.iloc[i], T)
        v_guess   = cca_base['V']       if cca_base['converged'] else (row['LCL_usd'] + row['B_f'])
        sig_guess = cca_base['sigma_V'] if cca_base['converged'] else (row['sigma_lcl'] * row['LCL_usd'] / v_guess)

        # B. Solve CCA under single OVX jump process (lam_base=0 collapses double sum to single)
        cca_jd = pricer.solve_CCA_jd(
            LCL_usd=row['LCL_usd'], sigma_lcl=row['sigma_lcl'],
            B_f=row['B_f'], r_f=r_f.iloc[i], T=T,
            lam_base=0, mu_base=0, sig_base=0,
            lam_ovx=lam_ovx_ann, mu_ovx=mu_ovx, sig_ovx=sigma_ovx,
            v_guess=v_guess, sig_guess=sig_guess
        )

        # C. Compute risk metrics from calibrated parameters
        if cca_jd['converged']:
            risk = pricer.compute_risk_jd(
                V=cca_jd['V'], sigma_diff=cca_jd['sigma_diff'],
                B_f=row['B_f'], r_f=r_f.iloc[i], T=T,
                lam_base=0, mu_base=0, sig_base=0,
                lam_ovx=lam_ovx_ann, mu_ovx=mu_ovx, sig_ovx=sigma_ovx
            )
            out['implied_V'].append(cca_jd['V'])
            out['implied_sigma_V'].append(cca_jd['sigma_total'])
            out['cca_converged'].append(True)
            out['distance_to_distress'].append(risk.get('d2', np.nan))
            out['default_prob'].append(risk.get('default_prob', np.nan))
            out['model_spread_bps'].append(risk.get('credit_spread_bps', np.nan))
            out['put_value'].append(risk.get('put_value', np.nan))
            out['risky_debt'].append(risk.get('risky_debt', np.nan))
            out['leverage'].append(risk.get('leverage', np.nan))
        else:
            for key in out:
                out[key].append(np.nan)
            out['cca_converged'][-1] = False

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])

print("Calibration complete!")


Starting M2 JD CCA Calibration loop...
Processing Abu Dhabi...
Processing Brazil...
Processing Chile...
Processing China...
Processing Colombia...
Processing Dubai...
Processing Egypt...
Processing Indonesia...
Processing Malaysia...
Processing Mexico...
Processing Philippines...
Processing Qatar...
Processing Saudi Arabia...
Processing South Africa...
Processing South Korea...
Processing Thailand...
Processing Turkey...
Calibration complete!


In [17]:
results.to_csv("../output/results/M2_results.csv")